In [2]:
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PythonLoader, JSONLoader
from langchain.schema import Document
import os
import glob

from langchain.vectorstores import Chroma
from langchain.text_splitter import PythonCodeTextSplitter
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PythonLoader
from langchain.schema import Document
from langchain.embeddings.base import Embeddings
from langchain_core.language_models.llms import LLM
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline
from langchain.embeddings.base import Embeddings
from langchain_huggingface import HuggingFacePipeline
import warnings
import os
import glob
import torch
import numpy as np
import subprocess
from typing import Any, List, Mapping, Optional
import warnings

from transformers import AutoTokenizer, AutoModel
# Suppress specific warning
warnings.filterwarnings("ignore", message="To copy construct from a tensor")

## Load documents

search and load python files 

In [3]:
all_docs = []

# find and load all python files to begin processing

scripts_dir = os.path.join("repo-exp", "scripts")
python_files = glob.glob(os.path.join(scripts_dir, "*.py"))

for file in python_files: 
    docs = PythonLoader(file).load()
    for doc in docs: 
        doc.metadata["source"] = file
    all_docs.extend(docs)
    
len(all_docs)

7

## Chunk files


PythonCodeTextSplitter: Use LangChain's specialized splitter that understands Python syntax and splits by functions, classes, and blocks

In [4]:
from langchain.text_splitter import PythonCodeTextSplitter
text_splitter = PythonCodeTextSplitter(chunk_size=1750, chunk_overlap=100)
texts = text_splitter.split_documents(all_docs)

In [5]:
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=100)

In [6]:
# if texts:
#     sample = texts[2]
#     print("\nSample chunk metadata:")
#     print(f"Source: {sample.metadata.get('source', 'Unknown')}")
#     print(f"Content preview: \n\n {sample.page_content}...\n")

## Initialize embeddings model

In [7]:
class CustomHuggingFaceEmbeddings(Embeddings):
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cpu"):
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.device = device
        self.model.to(device)
        # Set to evaluation mode
        self.model.eval()
    
    def _get_embedding(self, text):
        # Tokenize input text
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Get embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Mean pooling - take average of all token embeddings
            token_embeddings = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = sum_embeddings / sum_mask
            
        # Convert to numpy array and return
        return embeddings[0].cpu().numpy()
    
    def embed_documents(self, texts):
        """Generate embeddings for a list of documents."""
        return [self._get_embedding(text) for text in texts]
    
    def embed_query(self, text):
        """Generate embedding for a query."""
        return self._get_embedding(text)


In [8]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device set to use {device}")
embeddings = CustomHuggingFaceEmbeddings(device=device)

print("Creating vector database...")
# Create Chroma vector database
persist_directory = "chroma_db"
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)
vectordb.persist()
print(f"Vector database created and persisted to {persist_directory}")

Device set to use mps
Creating vector database...
Vector database created and persisted to chroma_db


/var/folders/2z/h5swrrf951g7z8vkhb3q8cy00000gn/T/ipykernel_10433/1848491000.py:13: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [14]:
model_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.25,
    repetition_penalty=1.2
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.
Device set to use mps


In [15]:
class MLXLLM(LLM):
    """LLM wrapper for MLX-powered Phi-3 model."""
    
    model_name: str = "microsoft/Phi-3-mini-4k-instruct"
    max_tokens: int = 256
    temperature: float = 0.25
    
    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        """Run MLX inference via command line."""
        cmd = [
            "python", "-m", "mlx_lm.generate",
            "--model", self.model_name,
            "--max-tokens", str(self.max_tokens),
            "--temp", str(self.temperature),
            "--prompt", prompt
        ]
        
        # Run the mlx_lm command
        result = subprocess.run(cmd, capture_output=True, text=True)
        output = result.stdout
        
        # Extract the generated response (adjust based on actual mlx_lm output format)
        try:
            # The output format might depend on mlx_lm, this is a basic approach
            # that assumes the response follows the prompt
            answer = output.split(prompt, 1)[1].strip()
            return answer
        except (IndexError, AttributeError):
            # Fallback to returning full output if parsing fails
            return output
    
    @property
    def _llm_type(self) -> str:
        return "mlx-phi-3"

In [16]:
llm = MLXLLM(
        model_name="microsoft/Phi-3-mini-4k-instruct",
        max_tokens=256,
        temperature=0.25
    )
    

In [17]:
def setup_qa_chain(vectordb, llm):
    """Set up a retrieval QA chain."""
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectordb.as_retriever(search_kwargs={"k": 4}),
        return_source_documents=True
    )
    return qa

In [24]:
query = "What does detect_garment() do in ol_label_images.py? What purpose does it serve?"
qa = setup_qa_chain(vectordb, llm)
ans = qa.invoke(query)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [25]:
def clean_qa_output(qa_response):
    # If the response is a dictionary with a 'result' key
    if isinstance(qa_response, dict) and 'result' in qa_response:
        return qa_response['result'].strip()
    # If the response is a string
    elif isinstance(qa_response, str):
        return qa_response.strip()
    # If it's some other type of object with content
    elif hasattr(qa_response, 'content'):
        return qa_response.content.strip()
    else:
        return str(qa_response)

In [26]:
formatted_ans = clean_qa_output(ans)
print(formatted_ans)

The `detect_garment()` function in `ol_label_images.py` is responsible for determining whether a given name corresponds to a garment by checking against a predefined list of clothing items. It serves the purpose of identifying and filtering out clothing items from a list of objects detected in an image.

Here's a breakdown of the function's behavior:

1. The function takes a single parameter, `name`, which is the name of the object to be evaluated.
2. It checks if the object's name is present in the predefined list of clothing items (`CLOTHING_ITEMS`).
3. If the object's name is found in the list, the function returns `True`, indicating that the object is a garment.
4. If the object's name is not found in the list, the function returns `False`, indicating that the object is not a garment.

This function is used to filter out clothing items from the list of detected objects in an image, allowing the pipeline to focus on processing only the clothing items.<|end|>
Prompt: 1093 tokens, 380

In [21]:
query = "can you explain what this file does: filter-coco-fashion.py (repo-exp/scripts/filter-coco-fashion.py)"
qa = setup_qa_chain(vectordb, llm)
ans = qa.invoke(query)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
def clean_qa_output(qa_response):
    # If the response is a dictionary with a 'result' key
    if isinstance(qa_response, dict) and 'result' in qa_response:
        return qa_response['result'].strip()
    # If the response is a string
    elif isinstance(qa_response, str):
        return qa_response.strip()
    # If it's some other type of object with content
    elif hasattr(qa_response, 'content'):
        return qa_response.content.strip()
    else:
        return str(qa_response)

In [23]:
formatted_ans = clean_qa_output(ans)
print(formatted_ans)

This Python script is designed to extract fashion-suitable images from the COCO dataset. It uses the argparse library to parse command-line arguments and perform two main filtering steps:

1. Filtering based on annotations: The script reads the annotations file (either instances_*.json or person_keypoints_*.json) and filters images based on the following criteria:
   - The person in the image should occupy a minimum normalized area (specified by the `--min-person-area` argument).
   - The person in the image should have a minimum number of visible keypoints (specified by the `--min-keypoints` argument).
   - The keypoints of the person in the image should have a minimum score (specified by the `--keypoint-score` argument).

The filtered images are saved in the specified output directory (specified by the `--output` argument).

2. Additional visual filtering (optional): If the `--visual-filter` argument is provided, the script applies an additional visual quality filtering step. The fil